In [2]:
import pandas as pd
from datetime import datetime


# ==========================================
# 1. READ INVOICE DATA
# ==========================================

df = pd.read_csv("invoice.csv")

print("\n========== INVOICE DATA ==========")

print(df.head())


# ==========================================
# 2. DATA CLEANING
# ==========================================

print("\n========== DATA CLEANING ==========")

print("\nMissing Values:")

print(df.isnull().sum())


print("\nDuplicate Rows:")

print(df.duplicated().sum())


# Remove duplicates

df = df.drop_duplicates()


# Remove missing values

df = df.dropna()


# Convert dates

df["Invoice_Date"] = pd.to_datetime(
    df["Invoice_Date"]
)

df["Due_Date"] = pd.to_datetime(
    df["Due_Date"]
)


# Convert numeric columns

df["Quantity"] = pd.to_numeric(
    df["Quantity"]
)

df["Unit_Price"] = pd.to_numeric(
    df["Unit_Price"]
)


# ==========================================
# 3. CALCULATE ITEM TOTAL
# ==========================================

df["Item_Total"] = (
    df["Quantity"]
    * df["Unit_Price"]
)


print("\n========== ITEM TOTAL ==========")

print(
    df[
        [
            "Invoice_Number",
            "Item",
            "Quantity",
            "Unit_Price",
            "Item_Total"
        ]
    ]
)


# ==========================================
# 4. CALCULATE INVOICE TOTAL
# ==========================================

invoice_totals = (
    df.groupby("Invoice_Number")[
        "Item_Total"
    ]
    .sum()
    .reset_index()
)


invoice_totals.rename(
    columns={
        "Item_Total": "Total_Amount"
    },
    inplace=True
)


# ==========================================
# 5. CUSTOMER DETAILS
# ==========================================

customer_details = (
    df.groupby("Invoice_Number")
    .agg({

        "Customer_Name": "first",

        "Customer_Email": "first",

        "Invoice_Date": "first",

        "Due_Date": "first"
    })
    .reset_index()
)


# ==========================================
# 6. CREATE CONSOLIDATED REPORT
# ==========================================

report = customer_details.merge(
    invoice_totals,
    on="Invoice_Number"
)


# ==========================================
# 7. IDENTIFY OVERDUE INVOICES
# ==========================================

# For demonstration we use today's date.
# You can also replace it with a fixed date.

today = pd.Timestamp.today().normalize()


report["Status"] = report[
    "Due_Date"
].apply(

    lambda date:
    "Overdue"
    if date < today
    else "Pending"
)


# ==========================================
# 8. DAYS OVERDUE
# ==========================================

report["Days_Overdue"] = report[
    "Due_Date"
].apply(

    lambda date:
    max(
        0,
        (today - date).days
    )
)


# ==========================================
# 9. DISPLAY REPORT
# ==========================================

print("\n========== CONSOLIDATED INVOICE REPORT ==========")

print(
    report.to_string(
        index=False
    )
)


# ==========================================
# 10. OVERDUE INVOICES
# ==========================================

overdue = report[
    report["Status"] == "Overdue"
]


print("\n========== OVERDUE INVOICES ==========")

if len(overdue) > 0:

    print(
        overdue[
            [
                "Invoice_Number",
                "Customer_Name",
                "Due_Date",
                "Total_Amount",
                "Days_Overdue"
            ]
        ].to_string(
            index=False
        )
    )

else:

    print(
        "No overdue invoices."
    )


# ==========================================
# 11. TOTAL INVOICE VALUE
# ==========================================

total_invoice_value = (
    report["Total_Amount"].sum()
)


print(
    "\nTotal Invoice Value: ₹",
    round(
        total_invoice_value,
        2
    )
)


# ==========================================
# 12. AVERAGE INVOICE VALUE
# ==========================================

average_invoice = (
    report["Total_Amount"].mean()
)


print(
    "Average Invoice Value: ₹",
    round(
        average_invoice,
        2
    )
)


# ==========================================
# 13. TOTAL INVOICES
# ==========================================

total_invoices = len(report)


print(
    "Total Invoices:",
    total_invoices
)


# ==========================================
# 14. OVERDUE COUNT
# ==========================================

overdue_count = len(overdue)


print(
    "Overdue Invoices:",
    overdue_count
)


# ==========================================
# 15. PENDING COUNT
# ==========================================

pending_count = len(
    report[
        report["Status"] == "Pending"
    ]
)


print(
    "Pending Invoices:",
    pending_count
)


# ==========================================
# 16. EXPORT FINAL REPORT
# ==========================================

report.to_csv(
    "consolidated_invoice_report.csv",
    index=False
)


print(
    "\n✅ Consolidated report exported!"
)

print(
    "File: consolidated_invoice_report.csv"
)


# ==========================================
# 17. EXPORT OVERDUE REPORT
# ==========================================

overdue.to_csv(
    "overdue_invoices.csv",
    index=False
)


print(
    "File: overdue_invoices.csv"
)


# ==========================================
# 18. AUTOMATED SUMMARY REPORT
# ==========================================

summary = {

    "Total Invoices":
        total_invoices,

    "Total Invoice Value":
        round(
            total_invoice_value,
            2
        ),

    "Average Invoice Value":
        round(
            average_invoice,
            2
        ),

    "Overdue Invoices":
        overdue_count,

    "Pending Invoices":
        pending_count
}


summary_df = pd.DataFrame(
    [summary]
)


summary_df.to_csv(
    "invoice_summary.csv",
    index=False
)


print(
    "File: invoice_summary.csv"
)


print(
    "\n🎉 INVOICE PROCESSING COMPLETED!"
)


========== INVOICE DATA ==========
  Invoice_Number Customer_Name    Customer_Email Invoice_Date    Due_Date  \
0         INV001  Rahul Sharma   rahul@gmail.com   2026-08-01  2026-08-15   
1         INV002   Priya Singh   priya@gmail.com   2026-08-05  2026-08-20   
2         INV003  Ananya Gupta  ananya@gmail.com   2026-08-10  2026-08-25   
3         INV004   Arjun Kumar   arjun@gmail.com   2026-08-12  2026-08-27   
4         INV005    Neha Verma    neha@gmail.com   2026-08-15  2026-08-30   

         Item  Quantity  Unit_Price  
0      Laptop         1       55000  
1       Mouse         2         800  
2    Keyboard         1        1500  
3     Monitor         1       12000  
4  Headphones         2        2500  

========== DATA CLEANING ==========

Missing Values:
Invoice_Number    0
Customer_Name     0
Customer_Email    0
Invoice_Date      0
Due_Date          0
Item              0
Quantity          0
Unit_Price        0
dtype: int64

Duplicate Rows:
0

========== ITEM TOTAL ====